# Slew and Center: how many tries, how long, and what helped?

A client's (NINA's) **Slew and Center** (SCC) as the driver logs it (`utility/scc_analysis.py`):

1. **GOTO** — `slewtocoordinates` to the target: the *blind goto*, aimed by the alignment alone.
2. **SYNC** — `synctocoordinates` to the plate solve: where that goto really landed. Its **miss** is the separation
   from the target to the solve.
3. **Correction GOTO** — the same target again, now with that sync in the alignment.
4. **SYNC** — and so on until a solve lands within the client's tolerance.

NINA syncs a solve outside its tolerance (and sometimes one inside it), so an SCC whose last goto has no sync ended
*within tolerance*; one that ends on a sync either landed within tolerance on that solve or stopped. A single goto with
one sync (e.g. an alignment point) is a blind goto too and is kept, as a one-goto sequence.

Each goto is one of three kinds: **blind** (the first goto to a new target), **correction** (the later gotos of an
SCC, each within 150 s of the one before) and **re-centre** (a goto to the same target minutes after the last
sequence — NINA centring again between exposures: its miss is mostly the tracking drift since, so it is kept apart
from the blind gotos).

Questions:

1. **Which SCCs are in the logs?** Per session, and the corrections in force.
2. **How many tries, and how long?**
3. **How did each SCC home in?** A dartboard per SCC: where each goto landed, in arcminutes from the target.
4. **How good is the blind goto?** By alignment model (MACv2, MACv1, off), Backlash Compensation (BLC), alignment
   points and slew length.
5. **How good are the correction gotos?** By BLC, and against what the alignment still had wrong at the sync before.
6. **Which corrections helped?** The two together, per configuration and per night.
7. **Do the misses lean one way?** A consistent direction points at backlash or an offset rather than noise.

Each step's configuration is what the log shows in force at that goto: *MACv2* (2026-10-11 on: M2/M3 offsets and M2
scale fitted with QUEST), *MACv1* (the earlier MAC), *OFF*; BLC *on* when the log has Backlash Compensation lines.

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import importlib, scc_analysis as sa
importlib.reload(sa)                                     # module changes without a kernel restart

PATTERN = '*.log'               # logs to read, e.g. 'alpaca.jdm_Beta8.0_*.log'
LOG_DIR = os.path.abspath(os.path.join('..', 'logs', 'archive'))
TOLERANCE = 1.0                 # arcmin: your client's centring tolerance (NINA: Options > Plate Solving > Error)
SAME_TARGET_ARCSEC = sa.SAME_TARGET_ARCSEC      # gotos this close are to the same target
SYNC_WINDOW_S = sa.SYNC_WINDOW_S                # a goto's solve is the first sync within this long
SCC = -1                        # the SCC section 3 shows first (index into the SCC table)
ONLY_MULTI = True               # section 3: only SCCs with a correction goto (2+ gotos)
TEMPLATE = 'plotly_dark'
CONFIG_COLOURS = {'MACv2 / BLC on': 'lightgreen', 'MACv2 / BLC off': 'deepskyblue', 'MACv1 / BLC off': 'orange',
                  'OFF / BLC off': 'gray', 'none yet / BLC off': 'dimgray'}

steps = []
for p in sorted(glob.glob(os.path.join(LOG_DIR, PATTERN))):
    ev, lat = sa.parse_events(p)
    st = sa.scc_steps(ev, lat, same_target_arcsec=SAME_TARGET_ARCSEC, sync_window_s=SYNC_WINDOW_S)
    if len(st):
        steps.append(st.assign(session=os.path.basename(p).replace('alpaca.', '').replace('.log', '')))
steps = pd.concat(steps, ignore_index=True) if steps else pd.DataFrame()
steps['mac'] = steps['mac'].fillna('none yet')
steps['config'] = steps['mac'] + ' / BLC ' + steps['blc']
steps['scc_key'] = steps['session'] + '#' + steps['scc'].astype(str)
summ = []
for s, g in steps.groupby('session', sort=False):
    x = sa.scc_summary(g)
    summ.append(x.assign(session=s, scc_key=[f'{s}#{i}' for i in x.index], config=g.groupby('scc')['config'].first().values))
summ = pd.concat(summ, ignore_index=True)
summ['converged'] = (summ['outcome'] == 'within tolerance') | (summ['last_solved_miss'] <= TOLERANCE)
print(f"{steps['session'].nunique()} sessions, {len(summ)} sequences ({int((summ['gotos'] >= 2).sum())} with a correction goto, "
      f"{int((summ['kind'] == 'recentre').sum())} re-centres), {len(steps)} gotos, {int(steps['synced'].sum())} solved: "
      + ', '.join(f'{k} {v}' for k, v in steps[steps['synced']]['kind'].value_counts().items()))

## 1. Which SCCs are in the logs?

One row per session: the configuration, the sequences (SCCs with a correction goto, and single goto + sync), the gotos
per SCC, how many converged (the last goto unsynced, or its last solve within `TOLERANCE`), and the median times:
goto to its solve, and the first goto to the last.

In [ ]:
per = summ.groupby('session', sort=False).agg(config=('config', lambda c: ', '.join(sorted(set(c)))),
                                             sccs=('gotos', lambda g: int((g >= 2).sum())), singles=('gotos', lambda g: int((g == 1).sum())),
                                             recentres=('kind', lambda k: int((k == 'recentre').sum())),
                                             gotos_median=('gotos', 'median'), converged_pct=('converged', lambda c: 100 * c.mean()),
                                             blind_miss=('first_miss', 'median'), to_solve_s=('to_solve_median_s', 'median'),
                                             to_last_goto_s=('to_last_goto_s', 'median'))
display(per.round(1))

## 2. How many tries, and how long?

**Left:** gotos per SCC (SCCs with a correction goto), by configuration. **Middle:** seconds from a goto to its solve
(slew, settle, exposure, solve). **Right:** seconds from the first goto to the last, against the number of gotos. The
final solve isn't in the driver log, so add one goto-to-solve time for the whole SCC.

In [ ]:
m = summ[summ['gotos'] >= 2]
fig = make_subplots(rows=1, cols=3, subplot_titles=['gotos per SCC', 'goto to its solve (s)', 'first goto to the last (s)'])
for cfg, d in m.groupby('config'):
    c = CONFIG_COLOURS.get(cfg, 'white')
    fig.add_trace(go.Histogram(x=d['gotos'], name=cfg, marker_color=c, opacity=0.75, xbins=dict(size=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=d['gotos'], y=d['to_last_goto_s'], mode='markers', marker=dict(color=c, size=7),
                             text=d['scc_key'], showlegend=False), row=1, col=3)
for cfg, d in steps[steps['synced']].groupby('config'):
    fig.add_trace(go.Box(y=d['to_solve_s'], name=cfg, marker_color=CONFIG_COLOURS.get(cfg, 'white'), showlegend=False,
                         boxpoints='outliers'), row=1, col=2)
fig.update_layout(barmode='overlay', height=450, width=1450, template=TEMPLATE)
fig.update_xaxes(title_text='gotos', row=1, col=1)
fig.update_xaxes(title_text='gotos', row=1, col=3)
fig.show()
t = m.groupby('config').agg(sccs=('gotos', 'size'), nights=('session', 'nunique'), gotos_mean=('gotos', 'mean'),
                            one_correction_pct=('gotos', lambda g: 100 * (g <= 2).mean()),
                            converged_pct=('converged', lambda c: 100 * c.mean()),
                            to_solve_s=('to_solve_median_s', 'median'), to_last_goto_s=('to_last_goto_s', 'median'))
display(t.round(1))

## 3. How did each SCC home in?

The target at the centre; each solved goto as a dart where it landed (arcmin), numbered in order and joined by the
path the SCC took. `FRAME` 'azalt': horizontal = ΔAz × cos(Alt), vertical = ΔAlt (as the motors see it); 'radec':
east = ΔRA × cos(Dec), north = ΔDec. Rings at 1', 5' and 10', the tolerance dashed. An unsolved last goto (within the
client's tolerance) is a star at the centre. Step through the SCCs with the slider; hover a dart for its BLC take-ups.

In [ ]:
FRAME = 'azalt'                 # 'azalt' or 'radec'
RINGS = [1, 5, 10]              # arcmin

def scc_list():
    s = summ if not ONLY_MULTI else summ[summ['gotos'] >= 2]
    return list(s['scc_key'])

def dart_xy(d):
    if FRAME == 'azalt' and d['miss_h'].notna().any():
        return d['miss_h'], d['miss_v'], 'ΔAz·cos(Alt)', 'ΔAlt'
    return d['miss_east'], d['miss_north'], 'ΔRA·cos(Dec) (east)', 'ΔDec (north)'

keys = scc_list()
if keys:
    active = SCC % len(keys)
    fig, counts, ranges, titles = go.Figure(), [], [], []
    for key in keys:
        d = steps[steps['scc_key'] == key]
        x, y, xl, yl = dart_xy(d)
        hov = [f"goto {r.step}: {r.miss:.1f}' after {r.to_solve_s:.0f}s<br>BLC take-ups {r.takeups}{' (gave up ' + r.giveups + ')' if r.giveups else ''}"
               f"{', rotated' if r.rotated else ''}<br>{r.mac}, {r.points:.0f} points" if r.synced else ''
               for r in d.itertuples()]
        tr = [go.Scatter(x=x, y=y, mode='lines+markers+text', text=d['step'].astype(str), textposition='top center',
                         marker=dict(size=11, color=CONFIG_COLOURS.get(d['config'].iloc[0], 'white')),
                         line=dict(color='gray', width=1), hovertext=hov, hoverinfo='text', name='solved gotos')]
        last = d.iloc[-1]
        if not last['synced']:
            tr.append(go.Scatter(x=[0], y=[0], mode='markers', marker=dict(symbol='star', size=16, color='yellow'),
                                 name=f"goto {last['step']}: within tolerance"))
        counts.append(len(tr))
        e = np.abs(np.r_[x.dropna(), y.dropna()]) if len(x.dropna()) else np.array([TOLERANCE])
        ranges.append(max(2.0, float(np.max(e))) * 1.2)
        s = summ[summ['scc_key'] == key].iloc[0]
        titles.append(f"{key} — {s['start']:%Y-%m-%d %H:%M:%S}: {s['kind']}, {s['gotos']} gotos, first miss "
                      f"{s['first_miss']:.1f}', {s['config']} ({xl} / {yl})")
        for t in tr:
            fig.add_trace(t)
    n_scc = len(fig.data)
    a = np.linspace(0, 2 * np.pi, 120)
    for r in RINGS + [TOLERANCE]:
        fig.add_trace(go.Scatter(x=r * np.cos(a), y=r * np.sin(a), mode='lines', hoverinfo='skip', showlegend=False,
                                 line=dict(color='yellow' if r == TOLERANCE else 'gray', width=0.8,
                                           dash='dash' if r == TOLERANCE else 'dot')))
    starts = np.cumsum([0] + counts[:-1])
    sl = []
    for i, (s0, n) in enumerate(zip(starts, counts)):
        vis = [False] * n_scc + [True] * (len(RINGS) + 1)
        vis[s0:s0 + n] = [True] * n
        sl.append(dict(method='update', label=str(i), args=[{'visible': vis}, {'title.text': titles[i],
                       'xaxis.range': [-ranges[i], ranges[i]], 'yaxis.range': [-ranges[i], ranges[i]]}]))
    for j, t in enumerate(fig.data[:n_scc]):
        t.visible = starts[active] <= j < starts[active] + counts[active]
    fig.update_layout(sliders=[dict(active=active, steps=sl, currentvalue=dict(prefix='SCC: '), pad=dict(t=50))],
                      xaxis=dict(range=[-ranges[active], ranges[active]], zeroline=True, title='arcmin'),
                      yaxis=dict(range=[-ranges[active], ranges[active]], zeroline=True, scaleanchor='x', title='arcmin'),
                      height=750, width=850, template=TEMPLATE, title=titles[active])
    fig.show()
    display(steps[steps['scc_key'] == keys[active]][['step', 'goto_time', 'synced', 'miss', 'miss_h', 'miss_v', 'to_solve_s',
                                                     'slew_arcmin', 'takeups', 'giveups', 'rotated', 'mac', 'points', 'sga_seed']].round(1))

## 4. How good is the blind goto?

Every blind goto with a solve (the first goto to a new target, of an SCC or a single goto + sync): its miss by configuration (one dot per goto, hover for
the session), then against the alignment points it had and the slew it made. The blind goto is the alignment model's
own accuracy plus whatever the goto left in the gears: BLC finishes each motor engaged the way it tracks, so the
slack (~2' M1, ~7' M2, ~5' M3) is taken up before the solve.

In [ ]:
b = steps[(steps['kind'] == 'blind') & steps['synced']]
order = [c for c in CONFIG_COLOURS if c in set(b['config'])]
fig = make_subplots(rows=1, cols=3, column_widths=[0.4, 0.3, 0.3],
                    subplot_titles=["blind miss by configuration (')", 'against alignment points', 'against the slew (deg)'])
for cfg in order:
    d = b[b['config'] == cfg]
    c = CONFIG_COLOURS.get(cfg, 'white')
    fig.add_trace(go.Box(y=d['miss'], name=f"{cfg}<br>{len(d)} gotos, {d['session'].nunique()} nights", marker_color=c,
                         boxpoints='all', jitter=0.4, pointpos=0, text=d['session'], showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=d['points'], y=d['miss'], mode='markers', marker=dict(color=c, size=6), name=cfg,
                             text=d['session']), row=1, col=2)
    fig.add_trace(go.Scatter(x=d['slew_arcmin'] / 60, y=d['miss'], mode='markers', marker=dict(color=c, size=6),
                             showlegend=False, text=d['session']), row=1, col=3)
fig.update_yaxes(type='log', title_text="miss (arcmin)")
fig.update_xaxes(title_text='alignment points', row=1, col=2)
fig.update_xaxes(title_text='slew (deg)', row=1, col=3)
fig.update_layout(height=520, width=1450, template=TEMPLATE)
fig.show()
display(b.groupby('config')['miss'].describe(percentiles=[0.25, 0.5, 0.75])[['count', '25%', '50%', '75%']]
        .join(b.groupby('config')['session'].nunique().rename('nights')).loc[order].round(1))

## 5. How good are the correction gotos?

A correction goto moves only a few arcminutes, to a target the alignment has just been synced on, so what it misses by
is mostly what the alignment still gets wrong there and what the move leaves in the gears.

* **Left:** correction misses by configuration.
* **Middle:** each solved goto's miss against the next goto's (the correction): below the diagonal, the SCC homed in.
  The ratio next / this is the fraction a correction leaves (0 = perfect; over 1, *diverged*: it landed further off).
* **Right:** the correction's miss against the alignment's residual at the sync before it (the driver seeds the Sync
  Guide offset, SGA, with it so the correction aims past it). Well under the diagonal: SGA carried the residual (the
  old QUEST-only alignments left ~30' there, and the corrections still landed within a few arcminutes). Once the
  residual is down to a few arcminutes (MACv2), the corrections sit at a floor set by other things — backlash,
  settling, the solve — and no longer follow it: `seed_corr` in the table is the correlation of the two.

In [ ]:
c = steps[(steps['kind'] == 'correction') & steps['synced']]
pairs = steps[steps['synced'] & steps['next_miss'].notna()]
order = [x for x in CONFIG_COLOURS if x in set(c['config'])]
fig = make_subplots(rows=1, cols=3, subplot_titles=["correction miss by configuration (')", "this miss -> the next goto's (')",
                                                    "correction miss vs the alignment's residual before it (')"])
for cfg in order:
    col = CONFIG_COLOURS.get(cfg, 'white')
    d = c[c['config'] == cfg]
    fig.add_trace(go.Box(y=d['miss'], name=f"{cfg}<br>{len(d)} gotos, {d['session'].nunique()} nights", marker_color=col,
                         boxpoints='all', jitter=0.4, pointpos=0, text=d['session'], showlegend=False), row=1, col=1)
    p = pairs[pairs['config'] == cfg]
    fig.add_trace(go.Scatter(x=p['miss'], y=p['next_miss'], mode='markers', marker=dict(color=col, size=6), name=cfg,
                             text=p['scc_key']), row=1, col=2)
    fig.add_trace(go.Scatter(x=p['sga_seed'], y=p['next_miss'], mode='markers', marker=dict(color=col, size=6),
                             showlegend=False, text=p['scc_key']), row=1, col=3)
lim = [0.1, max(1.0, float(np.nanmax(pairs[['miss', 'next_miss', 'sga_seed']].to_numpy())))]
for col in (2, 3):
    fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines', line=dict(color='gray', dash='dot'), showlegend=False), row=1, col=col)
    fig.update_xaxes(type='log', row=1, col=col)
fig.update_yaxes(type='log')
fig.update_xaxes(title_text="this goto's miss (')", row=1, col=2)
fig.update_xaxes(title_text="alignment residual at the sync (')", row=1, col=3)
fig.update_layout(height=520, width=1450, template=TEMPLATE)
fig.show()
t = c.groupby('config').agg(gotos=('miss', 'size'), nights=('session', 'nunique'), median=('miss', 'median'),
                            within_tol_pct=('miss', lambda m: 100 * (m <= TOLERANCE).mean()))
t = t.join(pairs.groupby('config').agg(ratio_median=('ratio', 'median'), residual_median=('sga_seed', 'median'),
                                       diverged_pct=('ratio', lambda r: 100 * (r > 1).mean()),
                                       next_over_residual=('next_miss', lambda m: float(np.median(m / pairs.loc[m.index, 'sga_seed']))),
                                       seed_corr=('next_miss', lambda m: float(np.corrcoef(m, pairs.loc[m.index, 'sga_seed'])[0, 1])
                                                  if len(m) > 2 else np.nan)))
display(t.loc[order].round(2))
cb = c[c['blc'] == 'on']
if len(cb):
    print('BLC on: correction misses by whether BLC took up slack during the goto (or a rotate before its solve)')
    display(cb.groupby(np.where(cb['takeups'] > 0, 'took up slack', 'no take-up'))['miss'].describe()[['count', '50%']].round(1))

## 6. Which corrections helped?

The blind and correction gotos side by side per configuration, and per night for the MACv2 nights. A configuration
seen on one or two nights is only an indication: the night's alignment (its M2/M3 offsets, `analyse_quest.ipynb`
section 7), the targets and the seeing all vary between nights. An A/B night — BLC switched on and off between
gotos to the same targets — separates BLC from the night.

In [ ]:
blind = steps[(steps['kind'] == 'blind') & steps['synced']]
corr = steps[(steps['kind'] == 'correction') & steps['synced']]
order = [x for x in CONFIG_COLOURS if x in set(steps['config'])]
tab = pd.DataFrame({'nights': steps.groupby('config')['session'].nunique(),
                    'blind gotos': blind.groupby('config')['miss'].size(), "blind median '": blind.groupby('config')['miss'].median(),
                    'correction gotos': corr.groupby('config')['miss'].size(), "correction median '": corr.groupby('config')['miss'].median(),
                    'gotos per SCC': summ[summ['gotos'] >= 2].groupby('config')['gotos'].mean()}).reindex(order)
display(tab.round(1))
fig = go.Figure()
for kind, d, sym in (('blind', blind, 'circle'), ('correction', corr, 'diamond')):
    g = d.groupby('config')['miss'].median().reindex(order)
    fig.add_trace(go.Bar(x=order, y=g, name=f'{kind} goto (median)', marker_color='orange' if kind == 'blind' else 'deepskyblue'))
fig.update_layout(barmode='group', height=420, width=1000, template=TEMPLATE, yaxis_type='log', yaxis_title="miss (arcmin)",
                  title='Median miss by configuration')
fig.show()
v2 = steps[steps['mac'] == 'MACv2']
if len(v2):
    print('MACv2 nights:')
    display(v2[v2['synced']].groupby(['session', 'config', 'kind'])['miss'].agg(['count', 'median']).unstack('kind').round(1))

## 7. Do the misses lean one way?

Each solved goto's miss in Az/Alt (horizontal, vertical), blind and correction, by configuration, with the mean (a
cross). Noise scatters around the centre; a consistent lean is something systematic: a correction that stops short
the same way each time (backlash: the motor turns, the output hasn't yet), or an offset the alignment hasn't absorbed.

In [ ]:
s = steps[steps['synced'] & steps['miss_h'].notna() & (steps['kind'] != 'recentre')]
order = [x for x in CONFIG_COLOURS if x in set(s['config'])]
fig = make_subplots(rows=1, cols=2, subplot_titles=["blind gotos (')", "correction gotos (')"])
for col, kind in ((1, 'blind'), (2, 'correction')):
    for cfg in order:
        d = s[(s['kind'] == kind) & (s['config'] == cfg)]
        if not len(d):
            continue
        c = CONFIG_COLOURS.get(cfg, 'white')
        fig.add_trace(go.Scatter(x=d['miss_h'], y=d['miss_v'], mode='markers', marker=dict(color=c, size=5, opacity=0.6),
                                 name=cfg, legendgroup=cfg, showlegend=col == 1, text=d['scc_key']), row=1, col=col)
        fig.add_trace(go.Scatter(x=[d['miss_h'].mean()], y=[d['miss_v'].mean()], mode='markers',
                                 marker=dict(symbol='x', size=16, color=c, line=dict(width=2)), legendgroup=cfg,
                                 showlegend=False), row=1, col=col)
for col, r in ((1, 30), (2, 10)):
    fig.update_xaxes(range=[-r, r], zeroline=True, title_text='ΔAz·cos(Alt)', row=1, col=col)
    fig.update_yaxes(range=[-r, r], zeroline=True, title_text='ΔAlt', row=1, col=col)
fig.update_layout(height=600, width=1300, template=TEMPLATE)
fig.show()
display(s.groupby(['kind', 'config']).agg(gotos=('miss', 'size'), mean_h=('miss_h', 'mean'), mean_v=('miss_v', 'mean'),
                                          sd_h=('miss_h', 'std'), sd_v=('miss_v', 'std')).round(1))